# CRbot: imitation learning v1 (Colab T4)

Trains the `crbot_il_v1` policy / value network on the shards built locally by
`tools/build_il_dataset.py` (VanguardX101/IL_Replay replays run through the CRbot engine).

**Before you run:** upload the whole `data/il_shards/` folder (all `shard_*.npz` + `manifest.json`) to
Google Drive as `MyDrive/CRbot_Training/datasets/il_shards/`. Runtime → Change runtime type → **T4 GPU**.

Model: a compact transformer (~2.9M parameters) over 1 global token, 5 hand tokens (4 slots + next card)
and up to 30 unit tokens, built from the 333-float observation (`backend_engine/brain/il_features.py`).
Four heads:
1. **gate**: wait vs play (logit)
2. **card slot**: 0..3, masked by what is affordable
3. **tile**: 4 × 576 logits (one 18×32 map per hand slot, masked by the legal deploy zone of that card)
4. **value**: win probability

Output: `crbot_il_v1.onnx` + `crbot_il_v1.json` (metrics, vocab, gate threshold) in
`MyDrive/CRbot_Training/models/crbot_il_v1/` and as a download. Put both files in
`CR-Pro-Coach/pretrained_weights/crbot_il_v1/` on the Mac; the app's **Neural (IL v1)** persona picks them up.

In [ ]:
# 1. Environment
import os, sys, json, math, time, glob, hashlib, random
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !nvidia-smi -L
    !pip -q install onnx onnxruntime onnxscript
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/CRbot_Training"
    DATA_DIR = f"{ROOT}/datasets/il_shards"
    OUT_DIR = f"{ROOT}/models/crbot_il_v1"
else:                                   # local smoke test: IL_SHARDS_DIR / IL_OUT_DIR
    DATA_DIR = os.environ.get("IL_SHARDS_DIR", "data/il_shards")
    OUT_DIR = os.environ.get("IL_OUT_DIR", "il_out")
os.makedirs(OUT_DIR, exist_ok=True)

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("device:", DEVICE, "| torch", torch.__version__, "| data:", DATA_DIR)

In [ ]:
# 2. Training settings (defaults are tuned for a T4 and ~100k-300k samples)
CFG = dict(
    d_model=256, n_heads=8, n_layers=4, d_ff=512, dropout=0.1,
    batch_size=512, epochs=int(os.environ.get("IL_EPOCHS", 20)), lr=3e-4, weight_decay=0.01, warmup_steps=500,
    w_gate=1.0, w_slot=1.0, w_tile=1.0, w_value=0.5,
    val_fraction=0.1, mirror_aug=True, hand_perm_aug=True,
    max_samples=int(os.environ.get("IL_MAX_SAMPLES", 0)),       # 0 = all (smoke tests use a small number)
)
print(CFG)

In [ ]:
# 3. Load shards (all in RAM: ~1.5 KB per sample) and split by battle (no battle in both train and val)
manifest = json.load(open(f"{DATA_DIR}/manifest.json"))
assert manifest["schema"] == "il_shards_v1", manifest.get("schema")
OBS_DIM, NUM_TILES, VOCAB = manifest["obs_dim"], manifest["num_tiles"], manifest["vocab"]
V = len(VOCAB)
print(f"manifest: {manifest['total_samples']:,} samples ({manifest['total_plays']:,} plays) from "
      f"{manifest['total_battles']:,} battles, obs_dim {OBS_DIM}, {V} cards")

files = sorted(glob.glob(f"{DATA_DIR}/shard_*.npz"))
assert files, f"no shards in {DATA_DIR}"
keys = ["obs", "gate", "card_slot", "tile_index", "value_target", "slot_mask", "tile_mask", "battle", "side"]
parts = {k: [] for k in keys}
for f in files:
    with np.load(f) as z:
        for k in keys:
            parts[k].append(z[k])
D = {k: np.concatenate(v) for k, v in parts.items()}
del parts
if CFG["max_samples"]:
    D = {k: v[:CFG["max_samples"]] for k, v in D.items()}
assert D["obs"].shape[1] == OBS_DIM
N = len(D["gate"])

# validation = battles whose id hashes into the last 10% (stable across rebuilds of the same shards)
bucket = np.array([int(hashlib.md5(str(int(b)).encode()).hexdigest(), 16) % 1000 for b in D["battle"]])
is_val = bucket < int(CFG["val_fraction"] * 1000)
tr_idx, va_idx = np.flatnonzero(~is_val), np.flatnonzero(is_val)
print(f"{N:,} samples: train {len(tr_idx):,} / val {len(va_idx):,} "
      f"({len(np.unique(D['battle'][is_val])):,} val battles); play share {D['gate'].mean():.2f}")

T = {
    "obs": torch.from_numpy(D["obs"]),
    "gate": torch.from_numpy(D["gate"].astype(np.float32)),
    "slot": torch.from_numpy(D["card_slot"].astype(np.int64)),
    "tile": torch.from_numpy(D["tile_index"].astype(np.int64)),
    "value": torch.from_numpy(D["value_target"]),
    "slot_mask": torch.from_numpy(D["slot_mask"]),
    "tile_mask": torch.from_numpy(D["tile_mask"]),            # packed uint8 [N, 72], unpacked per batch
}

In [ ]:
# 4. Observation layout (must match backend_engine/brain/il_features.py, OBS_VERSION il_obs_v1)
G_END = 8 + 12 + 4 + V + 1          # global + towers + opponent perception = 63
H0, HF, NH = G_END, 6, 5            # 5 hand tokens x 6 features (4 slots + next card)
U0, UF, NU = H0 + NH * HF, 8, 30    # 30 unit tokens x 8 features
assert U0 + NU * UF == OBS_DIM, (U0, OBS_DIM)
GW, GH = 18, 32
BITS = torch.tensor([128, 64, 32, 16, 8, 4, 2, 1], dtype=torch.uint8)


def unpack_tiles(packed: torch.Tensor) -> torch.Tensor:
    """uint8 [B, 72] -> bool [B, 576] (np.packbits order: most significant bit first)."""
    bits = (packed.unsqueeze(-1) & BITS.to(packed.device)) > 0
    return bits.reshape(packed.shape[0], -1)[:, :NUM_TILES]


# left-right mirror of the arena: x -> 18 - x, tiles gx -> 17 - gx, left / right princess towers swap
_tile_mirror = torch.arange(NUM_TILES).reshape(GH, GW).flip(1).reshape(-1)
_tower_perm = torch.arange(OBS_DIM)
for a, b in ((10, 12), (11, 13), (16, 18), (17, 19)):
    _tower_perm[a], _tower_perm[b] = b, a


def augment(obs, slot, tile, smask, tmask):
    B = obs.shape[0]
    dev = obs.device
    if CFG["mirror_aug"]:
        flip = torch.rand(B, device=dev) < 0.5
        o = obs[:, _tower_perm.to(dev)]
        xs = U0 + 2 + UF * torch.arange(NU, device=dev)
        present = o[:, U0::UF][:, :NU] > 0
        o[:, xs] = torch.where(present, 1.0 - o[:, xs], o[:, xs])
        obs = torch.where(flip[:, None], o, obs)
        mir = _tile_mirror.to(dev)
        tile = torch.where(flip & (tile >= 0), mir[tile.clamp(min=0)], tile)
        tmask = torch.where(flip[:, None], tmask[:, mir], tmask)
    if CFG["hand_perm_aug"]:
        perm = torch.argsort(torch.rand(B, 4, device=dev), dim=1)          # new slot j shows old slot perm[j]
        hand = obs[:, H0:H0 + 4 * HF].reshape(B, 4, HF)
        hand = torch.gather(hand, 1, perm[:, :, None].expand(B, 4, HF))
        obs = obs.clone()
        obs[:, H0:H0 + 4 * HF] = hand.reshape(B, -1)
        smask = torch.gather(smask, 1, perm)
        inv = torch.argsort(perm, dim=1)
        slot = torch.where(slot >= 0, torch.gather(inv, 1, slot.clamp(min=0)[:, None])[:, 0], slot)
    return obs, slot, tile, smask, tmask

In [ ]:
# 5. Model: token transformer with explicit attention (exports cleanly to ONNX)
class Block(nn.Module):
    def __init__(self, d, h, ff, p):
        super().__init__()
        self.h, self.dk = h, d // h
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.qkv, self.proj = nn.Linear(d, 3 * d), nn.Linear(d, d)
        self.ff = nn.Sequential(nn.Linear(d, ff), nn.GELU(), nn.Linear(ff, d))
        self.drop = nn.Dropout(p)

    def forward(self, x, key_bias):                     # key_bias [B, 1, 1, S]: 0 or -1e4
        B, S, D = x.shape
        q, k, v = self.qkv(self.ln1(x)).reshape(B, S, 3, self.h, self.dk).permute(2, 0, 3, 1, 4)
        att = torch.softmax(q @ k.transpose(-1, -2) / math.sqrt(self.dk) + key_bias, dim=-1)
        y = (self.drop(att) @ v).transpose(1, 2).reshape(B, S, D)
        x = x + self.drop(self.proj(y))
        return x + self.drop(self.ff(self.ln2(x)))


class CRbotIL(nn.Module):
    def __init__(self, vocab_size, d=256, h=8, layers=4, ff=768, p=0.1):
        super().__init__()
        self.V = vocab_size
        self.glob = nn.Sequential(nn.Linear(G_END, d), nn.GELU(), nn.Linear(d, d))
        self.card_emb = nn.Embedding(vocab_size + 1, d)       # 0 = empty slot
        self.hand_in = nn.Linear(HF, d)
        self.unit_in = nn.Sequential(nn.Linear(UF, d), nn.GELU(), nn.Linear(d, d))
        self.type_emb = nn.Parameter(torch.zeros(1 + 1 + NH + 1, d))   # cls, global, 5 hand, unit
        nn.init.normal_(self.type_emb, std=0.02)
        self.cls = nn.Parameter(torch.zeros(1, 1, d))
        self.blocks = nn.ModuleList(Block(d, h, ff, p) for _ in range(layers))
        self.ln = nn.LayerNorm(d)
        self.gate_head = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Linear(d, 1))
        self.value_head = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Linear(d, 1))
        self.slot_head = nn.Sequential(nn.Linear(2 * d, d), nn.GELU(), nn.Linear(d, 1))
        self.tile_query = nn.Sequential(nn.Linear(2 * d, d), nn.GELU(), nn.Linear(d, d))
        gy, gx = torch.meshgrid(torch.arange(GH), torch.arange(GW), indexing="ij")
        pos = torch.stack([gx.flatten() / (GW - 1), gy.flatten() / (GH - 1)], 1).float()
        freqs = torch.tensor([1.0, 2.0, 4.0, 8.0]) * math.pi
        four = torch.cat([torch.sin(pos[:, :, None] * freqs), torch.cos(pos[:, :, None] * freqs)], -1).flatten(1)
        self.register_buffer("tile_feats", torch.cat([pos, four], 1))        # [576, 18]
        self.tile_pos = nn.Linear(self.tile_feats.shape[1], d)
        self.tile_free = nn.Parameter(torch.zeros(NUM_TILES, d))
        self.tile_bias = nn.Parameter(torch.zeros(NUM_TILES))

    def forward(self, obs):
        B = obs.shape[0]
        g = self.glob(obs[:, :G_END])[:, None]
        hand = obs[:, H0:U0].reshape(B, NH, HF)
        idx = torch.clamp(torch.round(hand[:, :, 0] * (self.V + 1)), 0, self.V).long()
        h = self.hand_in(hand) + self.card_emb(idx)
        units = obs[:, U0:].reshape(B, NU, UF)
        u = self.unit_in(units)
        te = self.type_emb
        x = torch.cat([self.cls.expand(B, 1, -1) + te[0], g + te[1], h + te[2:2 + NH], u + te[2 + NH]], 1)
        pad = torch.cat([torch.zeros(B, 2 + NH, device=obs.device), (units[:, :, 0] <= 0).float()], 1)
        key_bias = (pad * -1e4)[:, None, None, :]
        for blk in self.blocks:
            x = blk(x, key_bias)
        x = self.ln(x)
        c, hs = x[:, 0], x[:, 2:6]                                     # cls, the 4 hand-slot tokens
        ch = torch.cat([c[:, None].expand(B, 4, -1), hs], -1)            # [B, 4, 2d]
        gate = self.gate_head(c)[:, 0]
        slot = self.slot_head(ch)[:, :, 0]
        q = self.tile_query(ch)                                        # [B, 4, d]
        tiles = self.tile_pos(self.tile_feats) + self.tile_free         # [576, d]
        tile = q @ tiles.t() / math.sqrt(q.shape[-1]) + self.tile_bias  # [B, 4, 576]
        value = torch.sigmoid(self.value_head(c)[:, 0])
        return gate, slot, tile, value


model = CRbotIL(V, CFG["d_model"], CFG["n_heads"], CFG["n_layers"], CFG["d_ff"], CFG["dropout"]).to(DEVICE)
print(f"parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

In [ ]:
# 6. Losses and validation metrics
NEG = -1e4


def batch_losses(out, gate_t, slot_t, tile_t, value_t, smask, tmask):
    gate, slot, tile, value = out
    play = gate_t > 0.5
    l_gate = F.binary_cross_entropy_with_logits(gate, gate_t)
    l_value = F.binary_cross_entropy(value.clamp(1e-6, 1 - 1e-6), value_t)
    if play.any():
        s_logits = slot[play].masked_fill(~smask[play], NEG)
        l_slot = F.cross_entropy(s_logits, slot_t[play])
        rows = tile[play].gather(1, slot_t[play][:, None, None].expand(-1, 1, NUM_TILES))[:, 0]
        t_logits = rows.masked_fill(~tmask[play], NEG)
        l_tile = F.cross_entropy(t_logits, tile_t[play])
    else:
        l_slot = l_tile = gate.sum() * 0
    total = CFG["w_gate"] * l_gate + CFG["w_slot"] * l_slot + CFG["w_tile"] * l_tile + CFG["w_value"] * l_value
    return total, dict(gate=l_gate.item(), slot=l_slot.item(), tile=l_tile.item(), value=l_value.item())


def get_batch(idx, train):
    b = {k: v[idx].to(DEVICE, non_blocking=True) for k, v in T.items()}
    tmask = unpack_tiles(b["tile_mask"])
    obs, slot, tile, smask = b["obs"], b["slot"], b["tile"], b["slot_mask"]
    if train:
        obs, slot, tile, smask, tmask = augment(obs, slot, tile, smask, tmask)
    return obs, b["gate"], slot, tile, b["value"], smask, tmask


@torch.no_grad()
def evaluate(idx_all, bs=2048):
    model.eval()
    acc = dict(n=0, gate_ok=0, plays=0, slot_ok=0, top3=0, dist=0.0, brier=0.0, ll=0.0, loss=0.0, batches=0)
    gate_scores, gate_labels = [], []
    for i in range(0, len(idx_all), bs):
        idx = torch.from_numpy(idx_all[i:i + bs])
        obs, gate_t, slot_t, tile_t, value_t, smask, tmask = get_batch(idx, train=False)
        out = model(obs)
        loss, _ = batch_losses(out, gate_t, slot_t, tile_t, value_t, smask, tmask)
        gate, slot, tile, value = out
        acc["loss"] += loss.item(); acc["batches"] += 1
        acc["n"] += len(idx)
        acc["gate_ok"] += ((gate > 0).float() == gate_t).sum().item()
        gate_scores.append(torch.sigmoid(gate).cpu()); gate_labels.append(gate_t.cpu())
        v = value.clamp(1e-6, 1 - 1e-6)
        acc["brier"] += ((v - value_t) ** 2).sum().item()
        acc["ll"] += -(value_t * v.log() + (1 - value_t) * (1 - v).log()).sum().item()
        play = gate_t > 0.5
        if play.any():
            p = play.nonzero()[:, 0]
            acc["plays"] += len(p)
            acc["slot_ok"] += (slot[p].masked_fill(~smask[p], NEG).argmax(1) == slot_t[p]).sum().item()
            rows = tile[p].gather(1, slot_t[p][:, None, None].expand(-1, 1, NUM_TILES))[:, 0]
            rows = rows.masked_fill(~tmask[p], NEG)
            top = rows.topk(3, dim=1).indices
            acc["top3"] += (top == tile_t[p][:, None]).any(1).sum().item()
            pred, true = top[:, 0], tile_t[p]
            dx = (pred % GW - true % GW).float(); dy = (pred // GW - true // GW).float()
            acc["dist"] += torch.sqrt(dx * dx + dy * dy).sum().item()
    n, k = acc["n"], max(acc["plays"], 1)
    scores, labels = torch.cat(gate_scores), torch.cat(gate_labels)
    best_thr, best_f1 = 0.5, -1.0
    for thr in np.arange(0.3, 0.91, 0.05):                       # gate threshold with the best F1 (for NeuralBot)
        pred = scores > thr
        tp = (pred & (labels > 0.5)).sum().item(); fp = (pred & (labels < 0.5)).sum().item()
        fn = (~pred & (labels > 0.5)).sum().item()
        f1 = 2 * tp / max(2 * tp + fp + fn, 1)
        if f1 > best_f1:
            best_thr, best_f1 = float(round(thr, 2)), f1
    model.train()
    return dict(loss=acc["loss"] / max(acc["batches"], 1), gate_acc=acc["gate_ok"] / n,
                slot_top1=acc["slot_ok"] / k, tile_top3=acc["top3"] / k, tile_dist=acc["dist"] / k,
                value_brier=acc["brier"] / n, value_logloss=acc["ll"] / n, gate_threshold=best_thr,
                gate_f1=best_f1)

# baselines on the validation set: always-first-affordable slot, most frequent label tile
va_play = va_idx[D["gate"][va_idx] == 1]
first_ok = D["slot_mask"][va_play].argmax(1)
base_slot = (first_ok == D["card_slot"][va_play]).mean() if len(va_play) else float("nan")
tr_play = tr_idx[D["gate"][tr_idx] == 1]
common_tiles = np.bincount(D["tile_index"][tr_play], minlength=NUM_TILES).argsort()[::-1][:3]
base_top3 = np.isin(D["tile_index"][va_play], common_tiles).mean() if len(va_play) else float("nan")
print(f"baselines (val): slot first-affordable {base_slot:.3f}, tile top-3 most-frequent {base_top3:.3f}")

In [ ]:
# 7. Train (AdamW + warmup / cosine, mixed precision on GPU); keep the best validation loss
opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
steps_per_epoch = max(len(tr_idx) // CFG["batch_size"], 1)
total_steps = steps_per_epoch * CFG["epochs"]
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / min(CFG["warmup_steps"], max(total_steps // 10, 1))) *
                                          0.5 * (1 + math.cos(math.pi * min(s / max(total_steps, 1), 1.0))))
use_amp = DEVICE == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
best, history = None, []
ckpt = f"{OUT_DIR}/crbot_il_v1_best.pt"
for epoch in range(CFG["epochs"]):
    t0 = time.time()
    perm = np.random.permutation(tr_idx)
    run = {"gate": 0.0, "slot": 0.0, "tile": 0.0, "value": 0.0}
    for step in range(steps_per_epoch):
        idx = torch.from_numpy(perm[step * CFG["batch_size"]:(step + 1) * CFG["batch_size"]])
        obs, gate_t, slot_t, tile_t, value_t, smask, tmask = get_batch(idx, train=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            out = model(obs)
        out = tuple(o.float() for o in out)
        loss, parts_ = batch_losses(out, gate_t, slot_t, tile_t, value_t, smask, tmask)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        for k in run:
            run[k] += parts_[k] / steps_per_epoch
    m = evaluate(va_idx)
    history.append({"epoch": epoch + 1, **{f"train_{k}": v for k, v in run.items()}, **m})
    flag = ""
    if best is None or m["loss"] < best["loss"]:
        best = {**m, "epoch": epoch + 1}
        torch.save({"model": model.state_dict(), "cfg": CFG, "vocab": VOCAB, "metrics": best}, ckpt)
        flag = "  *best*"
    print(f"epoch {epoch + 1:2d} {time.time() - t0:5.0f}s | train g {run['gate']:.3f} s {run['slot']:.3f} "
          f"t {run['tile']:.3f} v {run['value']:.3f} | val loss {m['loss']:.3f} gate {m['gate_acc']:.3f} "
          f"slot {m['slot_top1']:.3f} tile@3 {m['tile_top3']:.3f} dist {m['tile_dist']:.2f} "
          f"brier {m['value_brier']:.3f}{flag}")
print("best:", json.dumps(best, indent=1))

In [ ]:
# 8. Export the best checkpoint to ONNX and verify it with onnxruntime
import onnxruntime as ort
state = torch.load(ckpt, map_location="cpu")
export_model = CRbotIL(V, CFG["d_model"], CFG["n_heads"], CFG["n_layers"], CFG["d_ff"], 0.0)
export_model.load_state_dict(state["model"])
export_model.eval()
onnx_path = f"{OUT_DIR}/crbot_il_v1.onnx"
sample = T["obs"][torch.from_numpy(va_idx[:256])].float()
kw = dict(input_names=["obs"], output_names=["gate_logit", "slot_logits", "tile_logits", "value"],
          dynamic_axes={"obs": {0: "batch"}, "gate_logit": {0: "batch"}, "slot_logits": {0: "batch"},
                        "tile_logits": {0: "batch"}, "value": {0: "batch"}}, opset_version=17)
try:
    torch.onnx.export(export_model, (sample[:2],), onnx_path, dynamo=False, **kw)
except TypeError:                                   # older torch without the `dynamo` switch
    torch.onnx.export(export_model, (sample[:2],), onnx_path, **kw)

sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
ort_out = sess.run(None, {"obs": sample.numpy()})
with torch.no_grad():
    ref = [t.numpy() for t in export_model(sample)]
diffs = [float(np.abs(a - b).max()) for a, b in zip(ort_out, ref)]
print("onnxruntime vs torch max |diff| per output:", diffs)
assert max(diffs) < 1e-3, diffs
assert ort_out[2].shape == (len(sample), 4, NUM_TILES)

meta = {
    "name": "crbot_il_v1",
    "obs_version": manifest["obs_version"],
    "obs_dim": OBS_DIM,
    "vocab": VOCAB,
    "outputs": {"gate_logit": "[B] play logit", "slot_logits": "[B,4]", "tile_logits": "[B,4,576] gy*18+gx in the "
                "acting side's frame", "value": "[B] win probability"},
    "gate_threshold": best["gate_threshold"],
    "metrics": best,
    "baselines": {"slot_first_affordable": float(base_slot), "tile_top3_most_frequent": float(base_top3)},
    "train": {"samples": int(N), "train": int(len(tr_idx)), "val": int(len(va_idx)), "cfg": CFG,
              "dataset": manifest.get("source"), "filters": manifest.get("filters")},
    "history": history,
    "onnx_sha256": hashlib.sha256(open(onnx_path, "rb").read()).hexdigest(),
}
json.dump(meta, open(f"{OUT_DIR}/crbot_il_v1.json", "w"), indent=1)
print(f"saved {onnx_path} ({os.path.getsize(onnx_path) / 1e6:.1f} MB) + crbot_il_v1.json")

In [ ]:
# 9. Download (Colab): put both files in CR-Pro-Coach/pretrained_weights/crbot_il_v1/
if IN_COLAB:
    import shutil
    from google.colab import files
    pack = "/content/crbot_il_v1_pack"                 # only the .onnx + .json (no pickled .pt checkpoint)
    os.makedirs(pack, exist_ok=True)
    for name in ("crbot_il_v1.onnx", "crbot_il_v1.json"):
        shutil.copy(f"{OUT_DIR}/{name}", pack)
    files.download(shutil.make_archive("/content/crbot_il_v1", "zip", pack))